# 01 — Frozen-feature baselines
Bengali harmful meme detection: BHM, BanglaAbuseMeme, MIMOSA.
Image: CLIP ViT-L/14 · Text: BanglaBERT (mean pooling) · Classifier: Logistic Regression.
Kernel: **Python (internship1)**

## 1. Imports, seed, paths

In [ ]:
import os, gc, json, random, platform, numpy as np, pandas as pd, torch
import torch.nn.functional as Fn
from PIL import Image
from tqdm.auto import tqdm
import transformers, sklearn
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()

ROOT = "/mnt/f415dcc8-27a5-4e66-8493-25f1d939974c/internshipwork/journal"
P, FEAT = f"{ROOT}/processed", f"{ROOT}/features"
RES = f"{ROOT}/results/01_baselines_frozen"
os.makedirs(FEAT, exist_ok=True); os.makedirs(RES, exist_ok=True)
DATASETS = ["bhm", "banglaabusememe", "mimosa"]
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

## 2. Load encoders

In [ ]:
CLIP_NAME = "openai/clip-vit-large-patch14"
BERT_NAME = "csebuetnlp/banglabert"
MAX_LEN = 128

clip = CLIPModel.from_pretrained(CLIP_NAME, torch_dtype=torch.float16).to(dev).eval()
proc = CLIPProcessor.from_pretrained(CLIP_NAME)
tok  = AutoTokenizer.from_pretrained(BERT_NAME)
bert = AutoModel.from_pretrained(BERT_NAME).to(dev).eval()
print("models loaded")

## 3. Feature extraction functions

In [ ]:
@torch.no_grad()
def img_feats(paths, bs=32):
    out = []
    for i in tqdm(range(0, len(paths), bs), desc="image"):
        ims = [Image.open(p).convert("RGB") for p in paths[i:i + bs]]
        x = proc(images=ims, return_tensors="pt")["pixel_values"].to(dev, torch.float16)
        out.append(Fn.normalize(clip.get_image_features(pixel_values=x).float(), dim=-1).cpu())
    return torch.cat(out).numpy()

@torch.no_grad()
def txt_feats(texts, bs=64):
    out = []
    for i in tqdm(range(0, len(texts), bs), desc="text"):
        enc = tok(texts[i:i + bs], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(dev)
        h = bert(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1).clamp(min=1)).cpu())   # mean pooling
    return torch.cat(out).numpy()

## 4. Extract & save features (skips datasets already done)

In [ ]:
for ds in DATASETS:
    fi, ft = f"{FEAT}/{ds}_img.npy", f"{FEAT}/{ds}_txt.npy"
    if os.path.exists(fi) and os.path.exists(ft):
        print(ds, "already extracted - skipping")
        continue
    df = pd.read_csv(f"{P}/{ds}.csv").fillna("")
    print(f"\n{ds}: {len(df)} rows")
    np.save(fi, img_feats(df["image_path"].tolist()))
    np.save(ft, txt_feats(df["text"].astype(str).tolist()))
    print("saved", ds)

## 5. Free GPU memory

In [ ]:
del clip, bert
gc.collect(); torch.cuda.empty_cache()
print("GPU memory freed")

## 6. Load data + features

In [ ]:
data = {}
for ds in DATASETS:
    df = pd.read_csv(f"{P}/{ds}.csv").fillna("")
    img = np.load(f"{FEAT}/{ds}_img.npy")
    txt = np.load(f"{FEAT}/{ds}_txt.npy")
    assert len(df) == len(img) == len(txt), f"row mismatch in {ds}"
    data[ds] = {"df": df, "img": img, "txt": txt, "fus": np.hstack([img, txt])}
    print(ds, df.shape, "img", img.shape, "txt", txt.shape)

## 7. Classifier helpers (C tuned on validation split)

In [ ]:
C_GRID = [0.001, 0.01, 0.1, 1]

def fit_best(Xtr, ytr, Xva, yva):
    sc = StandardScaler().fit(Xtr)
    best_f, best_m = -1, None
    for C in C_GRID:
        m = LogisticRegression(C=C, max_iter=3000, class_weight="balanced",
                               random_state=SEED).fit(sc.transform(Xtr), ytr)
        f = f1_score(yva, m.predict(sc.transform(Xva)), average="macro")
        if f > best_f:
            best_f, best_m = f, m
    return sc, best_m, best_f

def split(ds, feat, label_col):
    d = data[ds]; df = d["df"]; X = d[feat]; y = df[label_col].values
    idx = {s: (df["split"] == s).values for s in ["train", "val", "test"]}
    return (X[idx["train"]], y[idx["train"]], X[idx["val"]], y[idx["val"]],
            X[idx["test"]], y[idx["test"]])

## 8. In-dataset baselines

In [ ]:
tasks = [("bhm", "label"), ("banglaabusememe", "label"),
         ("mimosa", "label"), ("mimosa", "label_bin")]
feats = [("txt", "Text (BanglaBERT)"), ("img", "Image (CLIP)"), ("fus", "Fusion (concat)")]

rows = []
for ds, lab in tasks:
    for feat, name in feats:
        Xtr, ytr, Xva, yva, Xte, yte = split(ds, feat, lab)
        sc, m, vf = fit_best(Xtr, ytr, Xva, yva)
        pred = m.predict(sc.transform(Xte))
        rows.append({"dataset": ds, "task": lab, "model": name, "best_C": m.C,
                     "val_f1": vf,
                     "test_f1": f1_score(yte, pred, average="macro"),
                     "test_acc": accuracy_score(yte, pred)})
        print(f"{ds:16s} {lab:10s} {name:18s} test macro-F1 = {rows[-1]['test_f1']:.4f}")

results = pd.DataFrame(rows)
table = results.pivot_table(index=["dataset", "task"], columns="model", values="test_f1").round(4)
table

## 9. Cross-dataset (binary harmful vs. not, fusion features)

In [ ]:
cross = []
for src in DATASETS:
    Xtr, ytr, Xva, yva, _, _ = split(src, "fus", "label_bin")
    sc, m, _ = fit_best(Xtr, ytr, Xva, yva)
    for tgt in DATASETS:
        _, _, _, _, Xte, yte = split(tgt, "fus", "label_bin")
        cross.append({"train_on": src, "test_on": tgt,
                      "macro_f1": f1_score(yte, m.predict(sc.transform(Xte)), average="macro")})

cross_df = pd.DataFrame(cross).pivot(index="train_on", columns="test_on", values="macro_f1").round(4)
cross_df

## 10. Save results + config

In [ ]:
results.to_csv(f"{RES}/baselines.csv", index=False)
table.to_csv(f"{RES}/baselines_table.csv")
cross_df.to_csv(f"{RES}/cross_dataset.csv")

config = {
    "seed": SEED,
    "clip": CLIP_NAME, "text_encoder": BERT_NAME,
    "text_pooling": "mean", "max_len": MAX_LEN,
    "classifier": f"LogisticRegression(class_weight=balanced), C tuned on val from {C_GRID}",
    "bam_split": "stratified 70/10/20, seed 42 (prepare_data.py)",
    "bam_text": "Surya OCR 0.17.1 (conf>=0.5) + EasyOCR fallback",
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sklearn": sklearn.__version__,
                 "gpu": torch.cuda.get_device_name(0) if dev == "cuda" else "cpu"},
}
with open(f"{RES}/config.json", "w") as f:
    json.dump(config, f, indent=2)
print("saved to", RES)